# DataFlow — Pipeline ETL
## Extracción → Transformación → Carga

**Fuente:** PostgreSQL (tablas Olist)  
**Destino:** MongoDB (colección `pedidos`)  

### Reglas de transformación documentadas
1. Calcular `dias_reales` y `dias_prometidos` desde timestamps de `orders`
2. Crear flag binario `tarde` = 1 si entrega real > entrega estimada
3. Imputar `review_score = 0` y textos vacíos cuando no hay reseña
4. Normalizar `category_name`: lowercase, reemplazar espacios por `_`
5. Calcular `volumen_cm3` = largo × alto × ancho
6. Excluir pedidos cancelados o sin timestamp de entrega (status != 'delivered')

In [1]:
# ── Librerías ──────────────────────────────────────────────────
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from pymongo import MongoClient
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv('../.env')
print('Librerías cargadas ✓')

Librerías cargadas ✓


## 1. EXTRACCIÓN — Leer desde PostgreSQL

In [2]:
# ── Conexión a PostgreSQL ──────────────────────────────────────
PG_URL = (
    f"postgresql://{os.getenv('POSTGRES_USER', 'dataflow_user')}"
    f":{os.getenv('POSTGRES_PASSWORD', 'dataflow_pass')}"
    f"@{os.getenv('POSTGRES_HOST', 'localhost')}"
    f":{os.getenv('POSTGRES_PORT', '5433')}"
    f"/{os.getenv('POSTGRES_DB', 'dataflow')}"
)

engine = create_engine(PG_URL)

with engine.connect() as conn:
    result = conn.execute(text('SELECT COUNT(*) FROM orders'))
    total = result.scalar()
    print(f'Conexion exitosa. Pedidos en PostgreSQL: {total:,}')

Conexion exitosa. Pedidos en PostgreSQL: 99,441


In [3]:
# ── JOIN maestro: extrae todos los datos necesarios ────────────
# order_payments no es tabla SQL del proyecto — se fusiona desde CSV en la siguiente celda

SQL_EXTRACT = """
SELECT
    o.order_id,
    o.order_status,
    o.purchase_timestamp,
    o.customer_delivery_timestamp,
    o.estimated_delivery_date,
    c.customer_unique_id,
    c.city         AS customer_city,
    c.state        AS customer_state,
    s.seller_id,
    s.city         AS seller_city,
    s.state        AS seller_state,
    p.product_id,
    p.category_name,
    COALESCE(p.weight_g, 0)                              AS weight_g,
    COALESCE(p.length_cm * p.height_cm * p.width_cm, 0) AS volume_cm3,
    oi.price,
    oi.freight_value,
    oi.item_seq,
    r.review_score,
    r.comment_title,
    r.comment_message
FROM orders o
JOIN customers    c  ON o.customer_id  = c.customer_id
JOIN order_items  oi ON o.order_id     = oi.order_id
JOIN products     p  ON oi.product_id  = p.product_id
JOIN sellers      s  ON oi.seller_id   = s.seller_id
LEFT JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.customer_delivery_timestamp IS NOT NULL
"""

print('Ejecutando extraccion...')
df_raw = pd.read_sql(SQL_EXTRACT, engine)
print(f'Filas extraidas: {len(df_raw):,}')
df_raw.head(3)

Ejecutando extraccion...


Filas extraidas: 110,485


,order_id,order_status,purchase_timestamp,customer_delivery_timestamp,estimated_delivery_date,customer_unique_id,customer_city,customer_state,seller_id,seller_city,...,product_id,category_name,weight_g,volume_cm3,price,freight_value,item_seq,review_score,comment_title,comment_message
0,00018f77f2f0320c557190d7a144bdd3,delivered,2017-04-26 10:53:06,2017-05-12 16:04:24,2017-05-15,eb28e67c4c0b83846050ddfb8a35d051,santa fe do sul,SP,dd7ddc04e1b6c2c614352b383efe2d36,sao paulo,...,e5f2d52b802189ee658865ca93d83a8f,pet_shop,30000.0,60000.0,239.9,19.93,1,4.0,None,None
1,000229ec398224ef6ca0657da4fc703e,delivered,2018-01-14 14:33:31,2018-01-22 13:19:16,2018-02-05,3818d81c6709e39d06b2738a8d3a2474,para de minas,MG,5b51032eddd242adc84c38acab88f23d,borda da mata,...,c777355d18b72b67abbeef9df44fd0fd,moveis_decoracao,3050.0,14157.0,199.0,17.87,1,5.0,None,Chegou antes do prazo previsto e o produto sur...
2,00048cc3ae777c65dbb7d2a0634bc1ea,delivered,2017-05-15 21:42:34,2017-05-22 13:44:35,2017-06-06,85c835d128beae5b4ce8602c491bf385,uberaba,MG,6426d21aca402a131fc0a5d0960a3c90,ribeirao preto,...,ef92defde845ab8450f9d70c526ef70f,utilidades_domesticas,450.0,2880.0,21.9,12.69,1,4.0,None,None


In [4]:
# ── Fusionar pagos desde CSV (no está en esquema SQL) ──────────
from pathlib import Path

df_pay = pd.read_csv(Path('../data/olist_order_payments_dataset.csv'))
df_pay = (
    df_pay.groupby('order_id')
    .agg(
        payment_type=('payment_type', 'first'),
        payment_installments=('payment_installments', 'max'),
        payment_value=('payment_value', 'sum')
    )
    .reset_index()
)

df_raw = df_raw.merge(df_pay, on='order_id', how='left')
df_raw['payment_type']         = df_raw['payment_type'].fillna('unknown')
df_raw['payment_installments'] = df_raw['payment_installments'].fillna(1).astype(int)
df_raw['payment_value']        = df_raw['payment_value'].fillna(0)

print(f'Pagos fusionados. Tipos de pago: {df_raw["payment_type"].value_counts().to_dict()}')

Pagos fusionados. Tipos de pago: {'credit_card': 83568, 'boleto': 22420, 'voucher': 2839, 'debit_card': 1655, 'unknown': 3}


## 2. TRANSFORMACIÓN — Limpieza y enriquecimiento

In [5]:
# ── Regla 1: Calcular días de entrega ─────────────────────────
df = df_raw.copy()

df['purchase_timestamp']          = pd.to_datetime(df['purchase_timestamp'])
df['customer_delivery_timestamp'] = pd.to_datetime(df['customer_delivery_timestamp'])
df['estimated_delivery_date']     = pd.to_datetime(df['estimated_delivery_date'])

df['dias_reales']     = (df['customer_delivery_timestamp'] - df['purchase_timestamp']).dt.days
df['dias_prometidos'] = (df['estimated_delivery_date']    - df['purchase_timestamp']).dt.days

print('Regla 1 aplicada: dias_reales y dias_prometidos calculados')
print(df[['dias_reales', 'dias_prometidos']].describe())

Regla 1 aplicada: dias_reales y dias_prometidos calculados


         dias_reales  dias_prometidos
count  110485.000000    110485.000000
mean       12.010327        23.443861
std         9.448929         8.824054
min         0.000000         2.000000
25%         6.000000        18.000000
50%        10.000000        23.000000
75%        15.000000        28.000000
max       209.000000       155.000000


In [6]:
# ── Regla 2: Flag de entrega tardía (TARGET del modelo) ────────
df['tarde'] = (df['customer_delivery_timestamp'] > df['estimated_delivery_date']).astype(int)

tasa_retraso = df['tarde'].mean() * 100
print(f'Regla 2 aplicada: flag "tarde" creado')
print(f'Tasa de retraso: {tasa_retraso:.1f}%')
print(df['tarde'].value_counts())

Regla 2 aplicada: flag "tarde" creado
Tasa de retraso: 7.9%
tarde
0    101750
1      8735
Name: count, dtype: int64


In [7]:
# ── Regla 3: Imputar reviews ausentes ─────────────────────────
# Pedidos sin reseña reciben score=0 y textos vacíos
sin_resena_antes = df['review_score'].isna().sum()

df['review_score']     = df['review_score'].fillna(0).astype(int)
df['comment_title']    = df['comment_title'].fillna('sin_reseña')
df['comment_message']  = df['comment_message'].fillna('')

print(f'Regla 3 aplicada: {sin_resena_antes:,} reseñas ausentes imputadas con score=0')

Regla 3 aplicada: 1,360 reseñas ausentes imputadas con score=0


In [8]:
# ── Regla 4: Normalizar categorías ────────────────────────────
def normalizar_categoria(cat):
    if pd.isna(cat):
        return 'sin_categoria'
    return (
        cat.lower()
           .strip()
           .replace(' ', '_')
           .replace('ó', 'o').replace('é', 'e')
           .replace('á', 'a').replace('ü', 'u')
    )

cat_antes = df['category_name'].nunique()
df['category_name'] = df['category_name'].apply(normalizar_categoria)
cat_despues = df['category_name'].nunique()

print(f'Regla 4 aplicada: {cat_antes} → {cat_despues} categorías únicas (normalización de texto)')
print('Muestra:', df['category_name'].value_counts().head(5).to_dict())

Regla 4 aplicada: 73 → 74 categorías únicas (normalización de texto)
Muestra: {'cama_mesa_banho': 11040, 'beleza_saude': 9481, 'esporte_lazer': 8455, 'moveis_decoracao': 8190, 'informatica_acessorios': 7669}


In [9]:
# ── Regla 5: Valores negativos o nulos en precios ─────────────
df['price']         = df['price'].clip(lower=0).fillna(0)
df['freight_value'] = df['freight_value'].clip(lower=0).fillna(0)
df['weight_g']      = df['weight_g'].clip(lower=0).fillna(0)
df['volume_cm3']    = df['volume_cm3'].clip(lower=0).fillna(0)
df['dias_reales']   = df['dias_reales'].clip(lower=0).fillna(0)
df['dias_prometidos'] = df['dias_prometidos'].clip(lower=1).fillna(1)

print('Regla 5 aplicada: valores negativos/nulos en métricas numéricas corregidos')

Regla 5 aplicada: valores negativos/nulos en métricas numéricas corregidos

In [10]:
# ── Resumen del estado del DataFrame transformado ─────────────
print('=== Estado final del DataFrame ===' )
print(f'Filas: {len(df):,}')
print(f'Columnas: {df.shape[1]}')
print('\nNulos por columna:')
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else 'Ninguno ✓')
print('\nTipos de datos:')
print(df.dtypes)

=== Estado final del DataFrame ===
Filas: 110,485
Columnas: 27

Nulos por columna:


Ninguno ✓

Tipos de datos:
order_id                               object
order_status                           object
purchase_timestamp             datetime64[ns]
customer_delivery_timestamp    datetime64[ns]
estimated_delivery_date        datetime64[ns]
customer_unique_id                     object
customer_city                          object
customer_state                         object
seller_id                              object
seller_city                            object
seller_state                           object
product_id                             object
category_name                          object
weight_g                              float64
volume_cm3                            float64
price                                 float64
freight_value                         float64
item_seq                                int64
review_score                            int64
comment_title                          object
comment_message                        object
payment

## 3. CARGA — Insertar documentos en MongoDB

In [11]:
# ── Conexión a MongoDB ─────────────────────────────────────────
MONGO_URI  = os.getenv('MONGO_URI', 'mongodb://localhost:27017/')
MONGO_DB   = os.getenv('MONGO_DB', 'dataflow')
MONGO_COL  = os.getenv('MONGO_COLLECTION', 'pedidos')

client = MongoClient(MONGO_URI)
db     = client[MONGO_DB]
col    = db[MONGO_COL]

# Limpiar colección antes de insertar (idempotente)
col.drop()
print(f'Colección "{MONGO_COL}" limpiada. Conexión MongoDB ✓')

Colección "pedidos" limpiada. Conexión MongoDB ✓


In [12]:
# ── Construir documentos anidados y cargar ────────────────────
def fila_a_documento(row):
    """Transforma una fila del DataFrame en documento MongoDB anidado."""
    return {
        '_id':           row['order_id'],
        'estado_pedido': row['order_status'],
        'fecha_compra':  str(row['purchase_timestamp']),
        'cliente': {
            'id':     row['customer_unique_id'],
            'ciudad': row['customer_city'],
            'estado': row['customer_state']
        },
        'vendedor': {
            'id':     row['seller_id'],
            'ciudad': row['seller_city'],
            'estado': row['seller_state']
        },
        'producto': {
            'id':        row['product_id'],
            'categoria': row['category_name'],
            'peso_g':    float(row['weight_g']),
            'volumen_cm3': float(row['volume_cm3'])
        },
        'pago': {
            'tipo':   row.get('payment_type', 'unknown'),
            'valor':  float(row.get('payment_value', row['price'] + row['freight_value'])),
            'cuotas': int(row.get('payment_installments', 1) or 1)
        },
        'entrega': {
            'dias_prometidos': int(row['dias_prometidos']),
            'dias_reales':     int(row['dias_reales']),
            'tarde':           bool(row['tarde'])
        },
        'reseña': {
            'score':  int(row['review_score']),
            'titulo': row['comment_title'],
            'texto':  row['comment_message']
        }
    }

# Inserción en lotes de 500 para eficiencia
BATCH_SIZE = 500
docs = df.apply(fila_a_documento, axis=1).tolist()

# Eliminar duplicados de order_id (un pedido puede tener múltiples ítems → tomar el primero)
seen = set()
docs_unicos = []
for d in docs:
    if d['_id'] not in seen:
        seen.add(d['_id'])
        docs_unicos.append(d)

print(f'Documentos únicos a insertar: {len(docs_unicos):,}')

insertados = 0
for i in tqdm(range(0, len(docs_unicos), BATCH_SIZE), desc='Cargando en MongoDB'):
    lote = docs_unicos[i:i+BATCH_SIZE]
    col.insert_many(lote)
    insertados += len(lote)

print(f'\n✅ Carga completada: {insertados:,} documentos insertados en MongoDB')

Documentos únicos a insertar: 96,470


Cargando en MongoDB:   0%|                                                                                                      | 0/193 [00:00<?, ?it/s]

Cargando en MongoDB:   4%|███▉                                                                                          | 8/193 [00:00<00:02, 69.08it/s]

Cargando en MongoDB:   9%|████████▏                                                                                    | 17/193 [00:00<00:02, 74.65it/s]

Cargando en MongoDB:  13%|████████████                                                                                 | 25/193 [00:00<00:02, 71.61it/s]

...

Cargando en MongoDB:  88%|████████████████████████████████████████████████████████████████████████████████▌           | 169/193 [00:02<00:00, 91.63it/s]

Cargando en MongoDB:  93%|█████████████████████████████████████████████████████████████████████████████████████▎      | 179/193 [00:02<00:00, 85.93it/s]

Cargando en MongoDB:  98%|██████████████████████████████████████████████████████████████████████████████████████████▌ | 190/193 [00:02<00:00, 91.29it/s]

Cargando en MongoDB: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 193/193 [00:02<00:00, 82.32it/s]


✅ Carga completada: 96,470 documentos insertados en MongoDB


In [13]:
# ── Verificación post-carga ────────────────────────────────────
total_mongo = col.count_documents({})
tardios     = col.count_documents({'entrega.tarde': True})
print(f'Total documentos en MongoDB: {total_mongo:,}')
print(f'Pedidos tardíos: {tardios:,} ({tardios/total_mongo*100:.1f}%)')
print('\nEjemplo de documento insertado:')
import json
doc_ejemplo = col.find_one({'entrega.tarde': True})
print(json.dumps(doc_ejemplo, indent=2, ensure_ascii=False, default=str))

Total documentos en MongoDB: 96,470
Pedidos tardíos: 7,826 (8.1%)

Ejemplo de documento insertado:
{
  "_id": "0005a1a1728c9d785b8e2b08b904576c",
  "estado_pedido": "delivered",
  "fecha_compra": "2018-03-19 18:40:33",
  "cliente": {
    "id": "639d23421f5517f69d0c3d6e6564cf0e",
    "ciudad": "santos",
    "estado": "SP"
  },
  "vendedor": {
    "id": "a416b6a846a11724393025641d4edd5e",
    "ciudad": "sao paulo",
    "estado": "SP"
  },
  "producto": {
    "id": "310ae3c140ff94b03219ad0adc3c778f",
    "categoria": "beleza_saude",
    "peso_g": 2000.0,
    "volumen_cm3": 5760.0
  },
  "pago": {
    "tipo": "credit_card",
    "valor": 157.6,
    "cuotas": 3
  },
  "entrega": {
    "dias_prometidos": 9,
    "dias_reales": 9,
    "tarde": true
  },
  "reseña": {
    "score": 1,
    "titulo": "sin_reseña",
    "texto": "Na descrição do produto quando fui efetuar a compra, estava detalhado os pentes de disfarce.\r\nQuando recebe o produto verifiquei e notei que não havia os pentes 0.5 e 1.5"